In [ ]:
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE, ADASYN
from ctgan import CTGAN
import os

# Load the dataset
path = os.path.join("dataset.csv")
dataset = pd.read_csv(path)

# Drop unnecessary columns
dataset = dataset.drop(['seqn', 'Marital'], axis='columns')

# Map categorical variables to numerical values
sex_mapping = {'Male': 0, 'Female': 1}
race_mapping = {'White': 0, 'Asian': 1, 'Black': 2, 'MexAmerican': 3, 'Hispanic': 4, 'Other': 5}
dataset['Sex'] = dataset['Sex'].replace(sex_mapping)
dataset['Race'] = dataset['Race'].replace(race_mapping)

# Fill NaN values with the mean of the respective columns
dataset.iloc[:, 2] = dataset.iloc[:, 2].fillna(dataset.iloc[:, 2].mean())
dataset.iloc[:, 4] = dataset.iloc[:, 4].fillna(dataset.iloc[:, 4].mean())
dataset.iloc[:, 5] = dataset.iloc[:, 5].fillna(dataset.iloc[:, 5].mean())

# Clean the dataset before splitting
dataset = dataset.dropna(subset=['MetabolicSyndrome'])

# Split the data into training and test sets
outcome_0 = dataset[dataset['MetabolicSyndrome'] == 0]
outcome_1 = dataset[dataset['MetabolicSyndrome'] == 1]
test_size_each_class = 400
test_0 = outcome_0.sample(n=test_size_each_class, random_state=42)
test_1 = outcome_1.sample(n=test_size_each_class, random_state=42)
test_data = pd.concat([test_0, test_1])
train_data = dataset.drop(test_data.index)

# Function to generate synthetic samples using SMOTE only
def generate_smote_samples(train_data):
    train_data = train_data.dropna(subset=['MetabolicSyndrome'])
    X = train_data.drop('MetabolicSyndrome', axis=1)
    y = train_data['MetabolicSyndrome']
    
    smote = SMOTE(random_state=42)
    X_smote, y_smote = smote.fit_resample(X, y)
    
    # Create dataframe with resampled data
    resampled_data = pd.DataFrame(X_smote, columns=X.columns)
    resampled_data['MetabolicSyndrome'] = y_smote
    
    return resampled_data

# Function to generate synthetic samples using CTGAN only
def generate_ctgan_samples(train_data):
    train_data = train_data.dropna(subset=['MetabolicSyndrome'])
    X = train_data.drop('MetabolicSyndrome', axis=1)
    y = train_data['MetabolicSyndrome']
    
    # Identify discrete (categorical) columns in your dataset
    discrete_columns = X.select_dtypes(include=['object', 'category']).columns.tolist()
    
    # Count samples needed to balance classes
    class_counts = train_data['MetabolicSyndrome'].value_counts()
    majority_count = class_counts.max()
    minority_count = class_counts.min()
    samples_needed = majority_count - minority_count
    
    # Generate synthetic samples using CTGAN
    ctgan = CTGAN(epochs=100)
    ctgan.fit(X, discrete_columns)
    
    # Generate only the number of samples needed for balancing
    ctgan_samples = ctgan.sample(samples_needed)
    ctgan_samples.columns = X.columns
    ctgan_samples['MetabolicSyndrome'] = 1  # Assuming minority class is 1
    
    # Combine original data with synthetic samples
    balanced_data = pd.concat([train_data, ctgan_samples])
    
    return balanced_data

# Function to generate synthetic samples using ADASYN only
def generate_adasyn_samples(train_data):
    train_data = train_data.dropna(subset=['MetabolicSyndrome'])
    X = train_data.drop('MetabolicSyndrome', axis=1)
    y = train_data['MetabolicSyndrome']
    
    adasyn = ADASYN(random_state=42)
    X_adasyn, y_adasyn = adasyn.fit_resample(X, y)
    
    # Create dataframe with resampled data
    resampled_data = pd.DataFrame(X_adasyn, columns=X.columns)
    resampled_data['MetabolicSyndrome'] = y_adasyn
    
    return resampled_data

# Function to generate combined synthetic samples (your original method)
def generate_combined_synthetic_samples(train_data, weights=(0.33, 0.33, 0.34)):
    train_data = train_data.dropna(subset=['MetabolicSyndrome'])

    X = train_data.drop('MetabolicSyndrome', axis=1)
    y = train_data['MetabolicSyndrome']

    # Identify discrete (categorical) columns in your dataset
    discrete_columns = X.select_dtypes(include=['object', 'category']).columns.tolist()
    
    # Generate synthetic samples using SMOTE
    smote = SMOTE(random_state=42)
    X_smote, y_smote = smote.fit_resample(X, y)
    smote_samples = pd.DataFrame(X_smote[len(X):], columns=X.columns)

    # Generate synthetic samples using ADASYN
    adasyn = ADASYN(random_state=42)
    X_adasyn, y_adasyn = adasyn.fit_resample(X, y)
    adasyn_samples = pd.DataFrame(X_adasyn[len(X):], columns=X.columns)

    # Generate synthetic samples using CTGAN
    ctgan = CTGAN(epochs=300)
    ctgan.fit(X, discrete_columns)
    ctgan_samples = ctgan.sample(len(adasyn_samples))
    ctgan_samples.columns = X.columns  # Ensure columns match original data

    # Combine synthetic samples based on weights
    n_smote = int(weights[0] * len(smote_samples))
    n_ctgan = int(weights[1] * len(ctgan_samples))
    n_adasyn = max(0, len(smote_samples) - n_smote - n_ctgan)

    final_synthetic = pd.concat([
        smote_samples.sample(n=n_smote, random_state=42, replace=True),
        ctgan_samples.sample(n=n_ctgan, random_state=42, replace=True),
        adasyn_samples.sample(n=n_adasyn, random_state=42, replace=True)
    ])

    final_synthetic['MetabolicSyndrome'] = 1  # Assuming synthetic samples are for the minority class
    balanced_data = pd.concat([train_data, final_synthetic])

    return balanced_data

# Function to train and evaluate model
def evaluate_model(train_data, test_data):
    X_train = train_data.drop('MetabolicSyndrome', axis=1).values
    y_train = train_data['MetabolicSyndrome'].values
    X_test = test_data.drop('MetabolicSyndrome', axis=1).values
    y_test = test_data['MetabolicSyndrome'].values
    
    model = XGBClassifier(random_state=42, n_estimators=100, learning_rate=0.3, max_depth=3)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    return {
        'accuracy': accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred),
        'recall': recall_score(y_test, y_pred),
        'f1': f1_score(y_test, y_pred)
    }

# Evaluate all methods and store results
def evaluate_all_methods(train_data, test_data):
    results = []
    
    # 1. Evaluate original unbalanced data
    original_results = evaluate_model(train_data, test_data)
    original_results['method'] = 'Original (Unbalanced)'
    results.append(original_results)
    
    # 2. Evaluate SMOTE
    smote_data = generate_smote_samples(train_data)
    smote_results = evaluate_model(smote_data, test_data)
    smote_results['method'] = 'SMOTE'
    results.append(smote_results)
    
    # 3. Evaluate CTGAN
    ctgan_data = generate_ctgan_samples(train_data)
    ctgan_results = evaluate_model(ctgan_data, test_data)
    ctgan_results['method'] = 'CTGAN'
    results.append(ctgan_results)
    
    # 4. Evaluate ADASYN
    adasyn_data = generate_adasyn_samples(train_data)
    adasyn_results = evaluate_model(adasyn_data, test_data)
    adasyn_results['method'] = 'ADASYN'
    results.append(adasyn_results)
    
    # 5. Evaluate Combined (with equal weights)
    combined_data = generate_combined_synthetic_samples(train_data)
    combined_results = evaluate_model(combined_data, test_data)
    combined_results['method'] = 'Combined (Equal Weights)'
    results.append(combined_results)
    
    # Convert results to DataFrame and save
    results_df = pd.DataFrame(results)
    results_df.to_csv('individual_synthetic_methods_results.csv', index=False)
    
    # Print results for quick viewing
    print(results_df[['method', 'accuracy', 'precision', 'recall', 'f1']])
    
    return results_df

# Evaluate all methods and get results
results_df = evaluate_all_methods(train_data, test_data)
